# M03 — 條件路由與 Command

本 notebook 對應 `README.md`，逐格執行即可。

主軸：把 M01／M02 那種「一條直線跑到底」的圖，
升級成會「看狀況分岔、會迴圈重試」的圖。
我們會做一個「依分數路由」的圖：答題 → 評分 → 達標就結束、不達標就退回重答（含次數上限），
最後用 `Command` 重寫其中一個 node，對比兩種寫法。

## 1. 環境準備

載入共用 helper，讓範例與供應商無關。本模組的重點是「圖的控制流」，
為了讓輸出可預期（環境不一定有 API key），評分先用一個確定性的假函式；
最後一格再示範如何換成真的 `model`。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
# We only build `model` here; the scoring demo below stays deterministic on purpose.
model = get_model()

## 2. 回顧 M01／M02：一條直線的圖

先回顧前面的做法：node 串成一條固定路線，每次都走同一條路。
這裡 `answer` 寫一個答案、`finalize` 收尾，中間沒有任何判斷。
注意 state 的設計（M02 學的）：我們之後要靠 `score` 和 `attempts` 來做路由。

In [ ]:
from typing import Annotated
from operator import add
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    topic: str                       # the question topic
    answer: str                      # latest answer text
    score: int                       # latest grade (0~100)
    attempts: Annotated[int, add]    # M02 reducer: accumulate the retry count


def answer(state: State) -> dict:
    # Pretend the model writes an answer; bump the attempt counter by 1.
    draft = f"關於「{state['topic']}」的第 {state['attempts'] + 1} 版答案"
    return {"answer": draft, "attempts": 1}   # reducer adds this 1 onto the total


def finalize(state: State) -> dict:
    return {"answer": state["answer"] + "（已定稿）"}


# A plain straight line: START -> answer -> finalize -> END.
straight = StateGraph(State)
straight.add_node("answer", answer)
straight.add_node("finalize", finalize)
straight.add_edge(START, "answer")
straight.add_edge("answer", "finalize")
straight.add_edge("finalize", END)
straight_graph = straight.compile()

print(straight_graph.invoke({"topic": "向量資料庫", "answer": "", "score": 0, "attempts": 0}))
# Expected output:
# {'topic': '向量資料庫', 'answer': '關於「向量資料庫」的第 1 版答案（已定稿）', 'score': 0, 'attempts': 1}

## 3. 加入評分 node（純函式、確定性）

真實情境會用模型或 rubric 打分。為了輸出可預期，這裡用一個確定性的假評分：
答案越長分數越高（每次重寫都會多帶一句，長度增加 → 分數提高）。
重點不是評分多聰明，而是它把 `score` 寫進 state，給後面的路由函式當判斷依據。

In [ ]:
def grade(state: State) -> dict:
    # Deterministic stand-in for a real grader: longer answer -> higher score.
    score = min(100, len(state["answer"]) * 3)
    print(f"[grade] attempt={state['attempts']} len={len(state['answer'])} -> score={score}")
    return {"score": score}


# Make `answer` append text so re-running it lengthens the answer (raises the score).
def answer(state: State) -> dict:
    extra = "，" + "補充" * state["attempts"] if state["attempts"] else ""
    draft = f"關於「{state['topic']}」的答案{extra}"
    return {"answer": draft, "attempts": 1}

## 4. 條件邊：依分數決定 pass 或 retry

核心來了。`route_by_score` 是一個「路由函式」：只讀 state、回傳一個 key 字串，**不改 state**。
`add_conditional_edges("grade", route_by_score, {...})` 把 key 對應到真正的目標 node。

迴圈的鐵律：一定要有終止條件。這裡同時檢查「達標了沒」和「次數到上限了沒」，
兩個出口（pass / give_up）都通往結束，只有 retry 會把邊指回前面的 `answer`。

In [ ]:
from typing import Literal

PASS_SCORE = 60
MAX_ATTEMPTS = 3


def route_by_score(state: State) -> Literal["pass", "retry", "give_up"]:
    # A pure "traffic cop": read state, point a direction, never modify state.
    if state["score"] >= PASS_SCORE:
        return "pass"
    if state["attempts"] >= MAX_ATTEMPTS:   # safety cap so the loop always stops
        return "give_up"
    return "retry"


builder = StateGraph(State)
builder.add_node("answer", answer)
builder.add_node("grade", grade)
builder.add_node("finalize", finalize)

builder.add_edge(START, "answer")
builder.add_edge("answer", "grade")          # always grade after answering
builder.add_conditional_edges(
    "grade",
    route_by_score,
    {
        "pass": "finalize",                  # good enough -> finish
        "retry": "answer",                   # not yet -> loop back to answer
        "give_up": "finalize",               # too many tries -> finish anyway
    },
)
builder.add_edge("finalize", END)
loop_graph = builder.compile()

## 5. 印 ASCII 圖：看見分支與迴圈

`draw_ascii()` 把圖畫成文字。注意看 `grade` 後面分出三條虛線（條件邊），
其中 `retry` 那條會指回 `answer`，形成迴圈。這比讀程式碼更快看懂流程長相。

In [ ]:
print(loop_graph.get_graph().draw_ascii())
# Expected output (示意，實際排版略有差異):
#         +-----------+
#         |  __start__|
#         +-----------+
#               *
#         +-----------+
#         |  answer    | <----+
#         +-----------+       |
#               *             | retry
#         +-----------+       |
#         |   grade    | -----+
#         +-----------+
#          *         *
#     pass /           \ give_up
#         +-----------+
#         | finalize   |
#         +-----------+
#               *
#         +-----------+
#         |  __end__   |
#         +-----------+

## 6. 跑迴圈：看它重試到達標

用 `stream(..., stream_mode="values")` 一步一步觀察 state 變化（M02 看過 values 模式）。
第一次答案太短分數不夠 → retry → 退回 answer 補字 → 再評分 → 直到達標或撞上限。
`attempts` 因為用了 `add` reducer，會一次次累加上去。

In [ ]:
init = {"topic": "向量資料庫", "answer": "", "score": 0, "attempts": 0}

for step in loop_graph.stream(init, stream_mode="values"):
    print("attempts=", step["attempts"], "| score=", step["score"], "| answer=", step["answer"])
# Expected output (示意):
# 每一輪 grade 都會印一行 [grade] ...，
# answer 變長、score 變高，attempts 從 0 一路累加，
# 直到 score >= 60 走 pass，最後一行的 answer 以「（已定稿）」結尾。

## 7. 驗證上限真的會擋下無限迴圈

把 `PASS_SCORE` 調到一個不可能達到的值（例如 9999），證明圖不會卡死：
它會重試到 `attempts` 撞上 `MAX_ATTEMPTS`，走 `give_up` 出口收尾。
這就是「上限是保命機制」的實證——沒有它，下面這段會永遠轉下去。

In [ ]:
def route_never_pass(state: State) -> Literal["pass", "retry", "give_up"]:
    # Same logic, but the pass bar is impossibly high to exercise the cap.
    if state["score"] >= 9999:
        return "pass"
    if state["attempts"] >= MAX_ATTEMPTS:
        return "give_up"
    return "retry"


capped = StateGraph(State)
capped.add_node("answer", answer)
capped.add_node("grade", grade)
capped.add_node("finalize", finalize)
capped.add_edge(START, "answer")
capped.add_edge("answer", "grade")
capped.add_conditional_edges(
    "grade", route_never_pass,
    {"pass": "finalize", "retry": "answer", "give_up": "finalize"},
)
capped.add_edge("finalize", END)
capped_graph = capped.compile()

final = capped_graph.invoke(init)
print("stopped at attempts =", final["attempts"])
# Expected output: stopped at attempts = 3  （撞上上限後 give_up，圖正常結束，不會無限迴圈）

## 8. 用 `Command` 重寫 grade：改 state + 跳轉一次到位

上面 `grade` 只「算分數」、把「往哪走」交給條件邊的路由函式。
但「往哪走」其實是 `grade` 算分時的自然產物——拆成兩處反而多繞一層。
用 `Command(update=..., goto=...)`：grade 自己算分、自己決定下一步，**這個 node 不用再接條件邊**。

In [ ]:
from langgraph.types import Command


def grade_cmd(state: State) -> Command:
    # Compute the score AND decide where to go, in one place.
    score = min(100, len(state["answer"]) * 3)
    if score >= PASS_SCORE:
        target = "finalize"
    elif state["attempts"] >= MAX_ATTEMPTS:
        target = "finalize"
    else:
        target = "answer"                    # loop back to retry
    print(f"[grade_cmd] attempt={state['attempts']} score={score} -> goto {target}")
    return Command(update={"score": score}, goto=target)


cmd_builder = StateGraph(State)
cmd_builder.add_node("answer", answer)
cmd_builder.add_node("grade", grade_cmd)     # this node routes itself via Command
cmd_builder.add_node("finalize", finalize)
cmd_builder.add_edge(START, "answer")
cmd_builder.add_edge("answer", "grade")
# Note: NO add_conditional_edges for "grade" — Command(goto=...) already owns the exit.
cmd_builder.add_edge("finalize", END)
cmd_graph = cmd_builder.compile()

print(cmd_graph.invoke(init)["answer"])
# Expected output: 一個以「（已定稿）」結尾的答案字串，行為和第 6 格的條件邊版本相同。

## 對比小結：條件邊 vs Command

| | 條件邊（第 4 格） | Command（第 8 格） |
|---|---|---|
| 做事 / 決定方向 | 分在 grade 和 route_by_score 兩處 | 都在 grade_cmd 一處 |
| 圖上要不要接條件邊 | 要 | 不要（goto 自己指定） |
| 何時較佳 | 路由規則獨立、會被多個 node 共用、好單獨測試 | 方向是這個 node 算出來的副產品，想少一層 |

兩者輸出相同。選哪個看「方向判斷該不該獨立存在」，而不是哪個比較潮。

## 🧪 練習 1：加一個「太長就精簡」的分支

目前只有「太短 → retry 補字」。請擴充路由：
當 `score == 100`（答案過長被打滿分視為冗長）時，走一個新的 node `trim`
把答案截短再回 `grade` 重評。

步驟提示：
1. 寫 `def trim(state): return {"answer": state["answer"][:20]}`，`add_node("trim", trim)`。
2. 路由函式多一個回傳值 `"too_long"`（記得用 `Literal` 把它列進去）。
3. 對照表加 `"too_long": "trim"`，並 `add_edge("trim", "grade")` 讓它回去重評。
4. 想一想：這條 trim→grade 也是迴圈，它的終止條件是什麼？會不會和 retry 互相打架？

In [ ]:
# TODO: 在這裡寫你的答案

## 🧪 練習 2：把 answer 也改成 Command 版

仿照第 8 格，把 `answer` 改寫成回傳 `Command`：
它一律 `goto="grade"`，但同時用 `update` 寫入新答案與 `attempts`。
改完後，這張圖的 `answer` 和 `grade` 都用 `Command` 自己決定走向，
圖上就只剩 `START → answer` 和 `finalize → END` 兩條普通邊。

想一想：當每個 node 都自己 goto，圖的「邊」幾乎消失了——
這樣比較好讀，還是反而看不出整體流程？（沒有標準答案，體會取捨即可。）

In [ ]:
# TODO: 在這裡寫你的答案
# def answer_cmd(state: State) -> Command:
#     extra = "，" + "補充" * state["attempts"] if state["attempts"] else ""
#     draft = f"關於「{state['topic']}」的答案{extra}"
#     return Command(update={"answer": draft, "attempts": 1}, goto="grade")

## 附錄：把確定性評分換成真模型

上面用假評分是為了輸出可預期。實務上 `grade` 會請模型打分。
概念示意如下（要有 API key 才能跑）——把模型回傳的分數寫進 `score`，
路由函式完全不用改，因為它只認 state 裡的 `score`。

In [ ]:
# from pydantic import BaseModel, Field
#
# class Grade(BaseModel):
#     score: int = Field(description="答案品質分數，0 到 100")
#
# grader = model.with_structured_output(Grade)
#
# def grade_with_model(state: State) -> dict:
#     result = grader.invoke(f"請為這個答案打分(0-100)：{state['answer']}")
#     return {"score": result.score}
#
# # 只要把 builder 裡的 grade node 換成 grade_with_model，其餘圖結構與路由原封不動。

## 小結 & 下一步

這個模組你做到了：

- 用 `add_conditional_edges` + 路由函式讓圖**分岔**：函式只讀 state、回傳 key，把 if/else 從 node 搬到邊上。
- 用「邊指回前面的 node」+「計數器上限」做出**有終止保證的重試迴圈**，並實證上限會擋下無限迴圈。
- 用 `Command(update=, goto=)` 把「改 state」和「決定走向」收進同一個 node，對比出它與條件邊各自的適用時機。
- 看 `draw_ascii()` 一眼讀懂分支與迴圈的形狀。

下一個模組 **M04 — 持久化與記憶**：到目前為止每次 `invoke` 都從零開始、跑完即忘。
M04 會用 `checkpointer` 加 `thread_id` 讓圖記得上一輪的 state，
這樣同一個 thread 的多次呼叫才會延續——也是 M05「中斷後續跑」的前提。